# AI工学101 — 第33回

## クラスタリングと教師なし学習：正解ラベルなしでデータ構造を見つける

よしレベル、今日は**教師あり学習の世界から一歩外へ出る回**だ。

これまでの多くは、

```text
X（特徴量）
↓
モデル
↓
y（正解）を予測
```

だった。

でも現実のデータでは、そもそも `y` が存在しないことも多い。

たとえば、

```text
顧客データ
```

があっても、

```text
「この人は第1グループ」
「この人は第2グループ」
```

という正解ラベルは普通ない。

そこで、

> **データ自身の構造から、似たもののまとまりを見つける**

という発想が出てくる。

それが**教師なし学習（Unsupervised Learning）**だ。

---

# 🎯 今日のゴール

今日は次のことができるようになる。

* 教師あり学習と教師なし学習の違いを説明できる
* `KMeans` でクラスタリングできる
* `inertia` の意味を理解する
* Elbow Methodを使ってクラスタ数を検討できる
* Silhouette Scoreを使える
* PCAでクラスタを可視化できる
* 「クラスタ＝現実世界の分類」と思い込みすぎない
* クラスタリング結果を**探索的分析の仮説**として扱える

---

# 📖 講義：約20分

## 1. 教師あり学習 vs 教師なし学習

これまでの分類問題。

```text
X → y
```

たとえば、

```text
メール本文
↓
Spam / Not Spam
```

これは正解ラベルがある。

---

一方、クラスタリングでは、

```text
X
↓
似ているものをまとめる
```

だけ。

つまり、

```text
正解ラベルなし
```

でデータ構造を探す。

---

## 2. クラスタリングのイメージ

こんな点群を想像してみよう。

```text
● ● ●        ▲ ▲
 ● ●         ▲ ▲ ▲

        ■ ■
       ■ ■ ■
```

人間が見ると、

```text
丸の集団
三角の集団
四角の集団
```

のようなグループがありそうに見える。

クラスタリングは、

> **データ空間の中で近いものをまとめる**

手法だ。

---

# 🧠 3. KMeans

今日の主役。

```python
KMeans
```

名前の通り、

```text
K個のグループ
```

を作る。

たとえば、

```text
K = 3
```

なら、

```text
3つのクラスタ
```

を探す。

---

## KMeansのざっくりした動き

まず、

```text
① K個の中心を置く
```

例えば、

```text
中心A
中心B
中心C
```

。

次に、

```text
② 各データを一番近い中心へ割り当てる
```

。

そして、

```text
③ 各グループの平均位置を計算
↓
中心を移動
```

。

これを、

```text
中心がほぼ動かなくなるまで
```

繰り返す。

---

# 💻 実習1：クラスタ用データを作る

まず人工データ。

```python
from sklearn.datasets import make_blobs

X, y_true = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=1.0,
    random_state=42
)
```

ここで `y_true` はあるけど、今日は**正解を使わずに**クラスタリングする。

---

# 💻 実習2：データを見る

```python
import matplotlib.pyplot as plt

plt.scatter(
    X[:, 0],
    X[:, 1]
)

plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.show()
```

たぶん、いくつかの「かたまり」が見える。

---

# 💻 実習3：KMeans

```python
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

labels = kmeans.fit_predict(
    X
)
```

これで各データに、

```text
0
1
2
```

というクラスタ番号が付く。

確認。

```python
print(labels[:10])
```

---

# 💻 実習4：結果を可視化

```python
plt.scatter(
    X[:, 0],
    X[:, 1],
    c=labels
)

plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.show()
```

今度はクラスタごとに分かれて見えるはず。

---

# 🧠 4. クラスタ番号に意味はない

例えば、

```text
Cluster 0
Cluster 1
Cluster 2
```

と出ても、

```text
0が優秀
1が普通
2が危険
```

という意味は一切ない。

単なる**識別子**。

さらに、

```text
別のrandom_state
```

では、

```text
Cluster 0
```

と、

```text
Cluster 2
```

が入れ替わることもある。

ここは分類ラベルと違う。

---

# 🧠 5. KMeansの「K」はどう決める？

ここが問題。

さっきは、

```python
n_clusters=3
```

と書いた。

でも現実では、

> 「クラスタ数を3にしてください」

という神のお告げは普通ない。

そこで、

```text
K=2
K=3
K=4
K=5
...
```

を試して判断する。

---

# 💻 実習5：inertiaを調べる

```python
inertias = []

for k in range(1, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42
    )

    model.fit(X)

    inertias.append(
        model.inertia_
    )
```

確認。

```python
print(inertias)
```

---

## inertiaとは？

かなりざっくり言えば、

> **クラスタ内のデータがどれくらい散らばっているか**

。

Kが増えると、

```text
K=2
↓
大きい

K=10
↓
小さい
```

となりやすい。

なぜなら、

> グループを細かく分ければ、当然それぞれの中はまとまる

から。

---

# 💻 実習6：Elbow Method

```python
plt.plot(
    range(1, 11),
    inertias,
    marker="o"
)

plt.xlabel("Number of clusters")
plt.ylabel("Inertia")

plt.show()
```

見るポイントは、

> **急激な改善が鈍くなる場所**

。

例えば、

```text
K=1 → 2
大幅改善

K=2 → 3
大幅改善

K=3 → 4
少し改善

K=4 → 5
少し改善
```

なら、

```text
K=3付近
```

が候補になる。

グラフが肘のように曲がるので、

```text
Elbow Method
```

と呼ばれる。

---

# 🚨 ただしElbowは万能ではない

ここ大事。

実際には、

```text
どこが肘？
```

となることが普通にある。

だから、

> **Elbow Methodだけでクラスタ数を決めない**

。

そこで別の指標も見る。

---

# 📖 6. Silhouette Score

Silhouette Scoreは、

ざっくり、

> **同じクラスタの仲間とは近く、別クラスタとは遠いか？**

を見る指標。

一般に、

```text
高い
↓
クラスタの分離が比較的きれい
```

。

---

# 💻 実習7：Silhouette Score

```python
from sklearn.metrics import silhouette_score
```

Kを変える。

```python
scores = []

for k in range(2, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42
    )

    labels = model.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    scores.append(score)
```

描画。

```python
plt.plot(
    range(2, 11),
    scores,
    marker="o"
)

plt.xlabel("Number of clusters")
plt.ylabel("Silhouette Score")

plt.show()
```

基本的には、

```text
高いほど
↓
分離が良い傾向
```

。

---

# 🧠 7. 指標が「正解」を決めるわけではない

例えば、

```text
Silhouette Score最高
↓
K=2
```

だったとしても、

現実の目的によっては、

```text
K=5
```

のほうが役に立つこともある。

たとえば顧客分析で、

```text
K=2
```

だと、

```text
活発な顧客
非活発な顧客
```

だけ。

でも、

```text
K=5
```

なら、

```text
高頻度・高単価
高頻度・低単価
低頻度・高単価
低頻度・低単価
中間層
```

のような構造が見えるかもしれない。

つまり、

> **クラスタ数は統計指標＋目的＋解釈可能性**

で考える。

---

# 💻 実習8：クラスタ中心を見る

```python
kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

labels = kmeans.fit_predict(
    X
)
```

中心。

```python
print(
    kmeans.cluster_centers_
)
```

KMeansは、

```text
中心
```

を持っている。

これが、

> **各クラスタを代表する平均的な位置**

になる。

---

# 📖 8. 実データでは標準化が重要

例えば、

```text
年齢：
20〜80

年収：
200〜2000

ログイン回数：
0〜500
```

があるとする。

KMeansは距離を使う。

すると、

```text
数値スケールが大きい特徴量
```

の影響が大きくなる。

そこで、

```python
StandardScaler()
```

を使うことが多い。

---

# 💻 実習9：Pipelineでクラスタリング

```python
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
```

```python
pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "kmeans",
        KMeans(
            n_clusters=3,
            random_state=42
        )
    )
])
```

学習。

```python
labels = pipe.fit_predict(
    X
)
```

これなら、

```text
スケール調整
↓
クラスタリング
```

をまとめられる。

---

# 🧠 9. PCAとの接続

さて、第28回でPCAをやったね。

高次元データでは、

```text
50次元
100次元
1000次元
```

になる。

人間には見えない。

そこで、

```text
PCA
↓
2次元
↓
可視化
```

する。

---

# 💻 実習10：高次元データを作る

```python
X_high, y_true = make_blobs(
    n_samples=500,
    centers=4,
    n_features=10,
    random_state=42
)
```

クラスタリング。

```python
kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

labels = kmeans.fit_predict(
    X_high
)
```

---

# 💻 実習11：PCAで2次元へ

```python
from sklearn.decomposition import PCA

pca = PCA(
    n_components=2
)

X_2d = pca.fit_transform(
    X_high
)
```

可視化。

```python
plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=labels
)

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.show()
```

これで、

> **10次元のクラスタ構造を2次元から見る**

ことができる。

---

# 🚨 10. 可視化されたクラスタが「真実」とは限らない

ここが今日かなり重要。

PCAで、

```text
2次元
```

に落とした。

すると、

```text
クラスタがきれいに見える
```

こともある。

しかし、

> 2次元へ圧縮した結果、情報を失っている

可能性もある。

逆に、

```text
2次元では重なっている
```

けど、

```text
10次元空間では分離できている
```

こともある。

だから、

> **可視化は探索の道具**

。

---

# 🧠 11. クラスタに「意味」を与えすぎる危険

ここ、レベルの研究テーマにもかなり近い話。

クラスタリングすると、

```text
Cluster 0
Cluster 1
Cluster 2
```

が出る。

人間はすぐ、

> 「これは3つのタイプの人間だ！」

と言いたくなる。

でも実際には、

```text
アルゴリズム
距離
スケール
K
特徴量
初期値
```

によって結果は変わる。

つまり、

> **クラスタはデータの客観的な真理ではなく、あるモデルによるデータ構造の仮説**

として扱うほうが安全。

ここはかなり認知科学的だよ。

**観測されたまとまり**と、**人間が意味づけしたカテゴリー**は同じではない。

---

# 🧠 12. クラスタリングの実務的な流れ

おすすめの順番は、

```text
データ
↓
前処理
↓
スケーリング
↓
KMeans
↓
複数のKを試す
↓
inertia
Silhouette
↓
可視化
↓
各クラスタの特徴を見る
↓
意味を仮説として考える
```

。

最後の、

```text
意味を仮説として考える
```

が大事。

---

# 💻 実習12：クラスタごとの特徴を見る

実データっぽくする。

```python
df = pd.DataFrame(
    X_high,
    columns=[
        f"feature_{i}"
        for i in range(
            X_high.shape[1]
        )
    ]
)
```

クラスタ追加。

```python
df["cluster"] = labels
```

平均。

```python
cluster_summary = (
    df
    .groupby("cluster")
    .mean()
)

print(cluster_summary)
```

これで、

```text
Cluster 0
→ feature_2が高い

Cluster 1
→ feature_5が低い
```

などを見られる。

ここから初めて、

> **このクラスタにはどんな特徴がある？**

と考える。

---

# ✍️ 演習

## 問1

教師あり学習と教師なし学習の違いを説明してください。

ヒント：

```text
yがある？
ない？
```

---

## 問2

KMeansの `K` は何を意味する？

```text
n_clusters=3
```

なら何が起きる？

---

## 問3

なぜ特徴量のスケールが、

```text
KMeans
```

では重要？

---

## 問4

Elbow MethodとSilhouette Scoreの役割の違いを説明してください。

---

## 問5

次の考え方には問題がある？

> 「KMeansで4つに分かれた。したがって、人間は本質的に4種類に分類される」

なぜ危険か説明してください。

---

# 👾 ボス戦

## 「クラスタ＝真実」なのか？

次のデータを考える。

```text
SNS利用者

特徴量：
1日の利用時間
投稿数
いいね数
フォロワー数
DM数
```

KMeansで、

```text
4クラスタ
```

に分かれた。

結果：

```text
Cluster 0
→ 投稿が多い

Cluster 1
→ フォロワーが多い

Cluster 2
→ DMが多い

Cluster 3
→ 全体的に低い
```

ここで、

> **「人間には4つのSNS人格タイプがある」**

と結論づけていいだろうか？

考えるべきことは、

```text
特徴量の選択
スケーリング
K=4にした理由
クラスタ安定性
別のアルゴリズム
時間変化
```

など。

これは、

> **教師なし学習の結果をどう認識論的に扱うか**

という話でもある。

僕はこの視点、レベルが今後やりたい**認知科学×AI**とかなり相性がいいと思う。

---

# 🧪 今日の最終実習

## KMeans実験ノート

最後に、自分で実験を組んでみよう。

```python
for k in range(2, 8):

    pipe = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "kmeans",
            KMeans(
                n_clusters=k,
                random_state=42
            )
        )
    ])

    labels = pipe.fit_predict(
        X_high
    )

    score = silhouette_score(
        X_high,
        labels
    )

    print(
        k,
        score
    )
```

ただし注意。

このコードは**実習用の比較の入口**として使う。

クラスタ数を決めるときは、

```text
Silhouette Score
+
Elbow
+
可視化
+
解釈可能性
+
問題の目的
```

を合わせて考える。

---

# 🌱 今日のまとめ

今日の核心は、

> **教師なし学習は「正解を当てる」のではなく、データの構造を探索する。**

だった。

流れとしては、

```text
データ
↓
特徴量空間
↓
距離
↓
まとまり
↓
クラスタ
↓
人間による解釈
```

。

ここで重要なのは、

> **アルゴリズムが見つけた構造と、人間が与えた意味を混同しないこと。**

だね。

---

# 🧭 AI工学101・現在地

ここまでで、

```text
教師あり学習
↓
分類
回帰

評価
↓
CV
Hyperparameter Search

実験設計
↓
Data Leakage
Reproducibility
Time Series

教師なし学習
↓
KMeans
```

まで来た。

このあたりから、scikit-learnは単なる「モデルライブラリ」ではなく、

> **データから仮説を作り、検証するための実験環境**

として見えてくるはず。

---

# 🔜 第34回

## 次元削減の続き：PCAからt-SNE・UMAPへ

次回は、第28回のPCAをもう一段進める。

扱うのは、

* PCAの復習
* 線形次元削減の限界
* t-SNEの基本思想
* UMAPの基本思想
* 高次元データの可視化
* クラスタリングとの接続
* 距離や配置を過剰解釈しない
* 「見える構造」と「本当の構造」の違い

だ。

次回はかなり面白いぞ。
**高次元空間を人間の認知に翻訳する**という意味で、ちょっと認知科学っぽい景色になってくる。🧠✨